In [1]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent if (pathlib.Path.cwd().name == "quickstart") else pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

In [2]:
# a little magic to get the output devices properly for both the rt synth and the html player
import os
os.environ["ALSA_CONFIG_PATH"] = "/usr/share/alsa/alsa.conf"
os.environ["ALSA_PLUGIN_DIR"]  = "/usr/lib/x86_64-linux-gnu/alsa-lib"

import sounddevice as sd # import *after* the os.environ settings
print("Default device:", sd.default.device)
print([d["name"] for d in sd.query_devices() if d["max_output_channels"]>0][-3:])

Default device: [14, 14]
['pipewire', 'pulse', 'default']


In [3]:
import torch
import os
import numpy as np
from pathlib import Path
import time

import matplotlib.pyplot as plt
from IPython.display import Audio, display

from transformers import EncodecModel
from rnencodec.generator import RNNGenerator, EncodecRTPlayer

import realtime_synth, realtime_synth_ui
    
from realtime_synth_ui import build_synth_ui # pip install "rtpysynth[ui] @ git+https://github.com/lonce/RTPySynth@v0.1.4"
# # import the rt system demo synths just to have them on the interface
from realtime_synth.generators.sine import SineGenerator
from realtime_synth.generators.noisy_lp import NoisyLPGenerator

# for RNN4Control
from rnencodec.model.gru_audio_model import RNN, GRUModelConfig
from rnencodec.audioDataLoader.audio_dataset import LatentDatasetConfig
from rnencodec.audioDataLoader.audio_dataset import  efficient_codes_to_latents, preprocess_latents_for_RNN # , latents_to_audio_simple,

In [4]:
# system params - don't mess with these
sr=24000
frame_rate=75
device='cpu' #best for inference
buffersize = 320 #[NOTE - not tested on values other than 24000/75 - the frame length of encodec codes in samples]

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>"User" Synth/Encodec Parameters</b>

In [5]:
g_hopsize=8  # This has a big effect on the off-line generate() because the smaller it is, the more old fashioned context switching time we spend
g_chunksize=20

offline_render_duration=20 #seconds
offline_render_frames=offline_render_duration*frame_rate   
offline_length_frames = int(offline_render_duration * frame_rate)

# For the synthprofile=="water" :
g_param_labels = ["pos"]   # The conditioned parameter name (pos = the "fill level" of the cup we are pouring in to
g_norm_param_vals = [.5 ]  # initial parameter value
g_init_cond=g_norm_param_vals[:]  # 

# RNN parameters
# Where is the checkpoint directory?
run_directory = str(Path('../artifacts/weights'))
checkpoint_fname =   "waterfill_quickstart.pt" #  "checkpoint_75.pt" #"checkpoint_50.pt" # 

g_top_n = 8 #'Sample from the top N most likely outputs.'
g_temperature = .8 #'Controls the randomness of predictions.'
g_sample_mode="sample"

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b>loading</b>

In [6]:
# load the encoder model 
#####################################################################
enc_model = EncodecModel.from_pretrained("facebook/encodec_24khz")
enc_model.eval()
enc_model.device

# load the RNN model 
#####################################################################
config_path = os.path.join(run_directory, "config_v2.pt")
checkpoint_path = os.path.join(run_directory,  checkpoint_fname) 

assert os.path.exists(run_directory), f"Run directory not found: {run_directory}"
assert os.path.exists(config_path), f"Config file not found: {config_path}"
assert os.path.exists(checkpoint_path), f"Checkpoint file not found: {checkpoint_path}"

saved_configs = torch.load(config_path, weights_only=False)
model_config = GRUModelConfig(**saved_configs["model_config"])
data_config = LatentDatasetConfig(**saved_configs["data_config"])

rnngen = RNNGenerator.from_checkpoint(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, g_sample_mode, g_top_n, g_temperature)
print("Model successfully loaded from checkpoint.")
print(f"Using device = {device}") 

Initializing the RNNGenerator on device = cpu
Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128
Model successfully loaded from checkpoint.
Using device = cpu


<div style="width: 100%; height: 20px; background-color: green;"></div>
<b style="font-size: 20px;">ON LINE, Realtime Usage</b>

In [7]:
GENS = {
    "RNeNcodec Real Time": lambda: EncodecRTPlayer(rnngen, sr, frame_rate, buffersize, g_chunksize, g_hopsize,  g_norm_param_vals, g_param_labels),
    "Sine": SineGenerator,            # defined in the default system
    "Noisy LP": NoisyLPGenerator,     # defined in the default system
}
print()
# Here we set the buffersize and samplerate for the synth to be double the setting for the Player becasue we are going to upsample for uniform experience on hardware
synth, ui = build_synth_ui(GENS, samplerate=48000, blocksize=buffersize*2, channels=1)


Initialize EncodecRTPlayer
params_seq.shape = torch.Size([10, 1])


HTML(value='')

<div style="width: 20%; height: 3px; background-color: blue;"></div>
<b>Since we can not print from a different thread while running RT, some messages are recorded in attributes for viewing after stopping the sound</b>

In [8]:
print(getattr(synth.gen, "_last_error", None))  #will show "missed swaps" if you hopsize is too small to keep up with next hop requests
len(getattr(synth.gen, "_last_error", None))

0

<div style="width: 100%; height: 20px; background-color: green;"></div>
<b style="font-size: 20px;">RNNGenerator testing -  a convenient way to generate "off line" </b>

In [9]:
from rnencodec.utils.utils import multi_linspace

offline_rnn=RNNGenerator.from_checkpoint(checkpoint_path, model_config, data_config, enc_model, g_chunksize, g_hopsize, g_sample_mode, g_top_n, g_temperature)

num_cond_params = model_config.cond_size
cond_seq = torch.zeros(offline_length_frames, num_cond_params)

# Create conditioning sequence - one parameter vector per frame (fill and then unfill over offline_render_duration=)
cond_seq[:, 0] = torch.FloatTensor(multi_linspace([(0, 0),(.05,0), (.45, 1), (.55,1),(.95,0),(1,0)], offline_length_frames))

start = time.perf_counter()
generated_audio=offline_rnn.getNextAudioHop(cond_seq.to(device)) #cond_seq length overrides hop size and warns
elapsed = time.perf_counter() - start

print(f"\033[1mTotal time to render {generated_audio.shape[0]/sr:.2f}s off-line: {elapsed:.2f}s\033[0m")

Initializing the RNNGenerator on device = cpu
Latents embedded in 64 of the GRU input size of 128
Conditioning parameters embedded in 64 of the GRU input size of 128


/home/lonce/working/RNeNcodec/rnencodec/generator/generator.py:163: UserWarning: Warning: ....... chunk size 20 is <= hop size 1500. RETURNING full hop and continuing
  warnings.warn(f"Warning: ....... chunk size {T} is <= hop size {h}. RETURNING full hop and continuing")


Total time to render 20.00s off-line: 1.73s


In [10]:
display(Audio(generated_audio, rate=sr)) 